

# Webscrapping com dados públicos: compras de materiais didáticos pelo MEC em 2025
Fonte: Diário Oficial da União <br><br>
Projeto elaborado com as PyLadies SP no Open Data Day, em 07/03/2026, usando Python, Selenium e Docker





O código, em Python, foi executado no Google Colab, o que permite que qualquer pessoa rode os exemplos diretamente no navegador, sem precisar instalar Python ou bibliotecas localmente.

No entanto, algumas ferramentas utilizadas no projeto — como Selenium e o navegador Chrome — podem exigir configurações específicas do sistema. Para facilitar a reprodução do ambiente de execução, o repositório também inclui uma configuração com Docker.

Docker permite criar um ambiente isolado com todas as dependências necessárias já instaladas, garantindo que o código funcione da mesma forma em qualquer computador.

Assim, você pode utilizar este material de duas formas:

1. Executar o notebook diretamente no Google Colab (opção mais simples)
2. Rodar o projeto localmente usando Docker, caso queira reproduzir o ambiente completo disponível [aqui](https://github.com/anacarolcortez/workshop-webscraping-docker)

Abaixo, vamos ver a opção nº 1 que foi desenvolvida durante o workshop, mas também fiz adições, como: a função lambda para formatar os valores em reais (R$); perguntas no tópico "6. Análise de Dados"; comentários extras ao longo do código; e a possibilidade de baixar o dataset limpo em .csv para ser usado em ferramentas de visualização.


#Objetivo deste projeto
Automatizar tarefas de coleta de dados na web, incluindo:


* navegação automatizada em páginas web;
* interação com elementos da página (botões, texto);
* extração de informações estruturadas;
* como lidar com esperas implícitas e explícitas;
* boas práticas para não ser bloqueada(o);
* organização dos dados para análise.

Vamos usar como exemplo o Diário Oficial da União buscando pela palavra-chave "materiais didáticos", e usar filtros para selecionar contratos firmados com o Ministério da Educação em 2025.

Vamos salvar as informações de cada contrato obtido em um arquivo no formato csv, ideal para futura análise de dados.

A raspagem de dados vai consistir em capturar elementos no DOM do HTML, interação com eles (clicar, inserir textos, mudar de página, copiar texto).

##Executando o código diretamente no Colab

### 1. Instalação das bibliotecas

In [3]:
# Instala o Google Chrome oficial
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt install -y ./google-chrome-stable_current_amd64.deb

# Instala o webdriver-manager para gerenciar o driver automaticamente
!pip install selenium webdriver-manager

--2026-03-13 02:44:41--  https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
Resolving dl.google.com (dl.google.com)... 74.125.126.190, 74.125.126.136, 74.125.126.93, ...
Connecting to dl.google.com (dl.google.com)|74.125.126.190|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 127504384 (122M) [application/x-debian-package]
Saving to: ‘google-chrome-stable_current_amd64.deb.1’

google-chrome-stabl 100%[===================>] 121.60M   173MB/s    in 0.7s    

2026-03-13 02:44:42 (173 MB/s) - ‘google-chrome-stable_current_amd64.deb.1’ saved [127504384/127504384]

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'google-chrome-stable' instead of './google-chrome-stable_current_amd64.deb'
google-chrome-stable is already the newest version (146.0.7680.75-1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [4]:
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager #faz a gestão do driver automaticamente
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

###2. Automação de consulta no DOU

In [5]:
# Configurações para rodar no ambiente do Colab
chrome_options = Options()
chrome_options.add_argument('--headless') # Sem interface gráfica
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--remote-debugging-port=9222') # Ajuda na estabilidade no Colab

# O WebDriverManager baixa e configura o driver compatível automaticamente
service = Service(ChromeDriverManager().install())

try:
    driver = webdriver.Chrome(service=service, options=chrome_options)

    print("Abrindo o browser...")
    driver.get("https://in.gov.br/acesso-a-informacao/dados-abertos/base-de-dados")

    print(f"Página carregada: {driver.title}")

except Exception as e:
    print(f"Erro:\n{e}")

Abrindo o browser...
Página carregada: Base de Dados de Publicações do DOU - Imprensa Nacional


In [6]:
#Buscar documentos por palavra-chave
#input_pesquisa = driver.find_element("id", "search-bar")

In [7]:
#Configurar um tempo de espera para o carregamento dos elementos no DOM
wait = WebDriverWait(driver, 30) #faz a gestão para tempo de espera de carregamento da página em 30s

input_pesquisa = wait.until(
    EC.presence_of_element_located((By.ID, "search-bar"))
)
input_pesquisa.clear()
input_pesquisa.send_keys("material didático")

In [8]:
#Clicar em Pesquisa Avançada
btn_pesquisa_avancada = wait.until(
    EC.element_to_be_clickable((By.ID, "toggle-search-advanced"))
)
btn_pesquisa_avancada.click()

In [9]:
#Selecionar a opção "Resultado Exato"
input_resultado_exato = wait.until(
    EC.element_to_be_clickable((
        By.ID, "tipo-pesquisa-1"
    ))
)
input_resultado_exato.click()

In [10]:
#Selecionar a opção "personalizado" em 'Data'
input_personalizado = wait.until(
    EC.element_to_be_clickable((
        By.ID, "personalizado"
    ))
)
input_personalizado.click()

In [11]:
###o código abaixo é só didático, pois o Colab não consegue rodar igual no Chrome para selecionar as datas
'''#Clicar no input "início"
input_data_inicio = wait.until(
    EC.element_to_be_clickable((
        By.ID, "data-inicio"
    ))
)
input_data_inicio.click()

#Escolhe a data 01/01/2025 no calendário
for i in range(0,14): #botão vai clicar 14 vezes para voltar de hoje (março de 2026) até o mês de janeiro de 2025
    btn_anterior = wait.until(
        EC.element_to_be_clickable((
            By.XPATH, "//a[@data-handler='prev']"
        ))
    )
    btn_anterior.click()

data_inicial = wait.until(
    EC.element_to_be_clickable((By.XPATH, "//a[@class='ui-state-default' and normalize-space(text())='1']"))
)
data_inicial.click()

#Clicar no input "fim"
input_data_fim = wait.until(
    EC.element_to_be_clickable((
        By.ID, "data-fim"
    ))
)
input_data_fim.click()

#Escolher a data 31/12/2025 no calendário
for i in range(0, 3): #já aqui o botão clica 3 vezes para voltar de março de 2026 (hoje) até dezembro de 2025
    btn_anterior = wait.until(
        EC.element_to_be_clickable((
            By.XPATH, "//a[@data-handler='prev']"
        ))
    )
    btn_anterior.click()

data_final = wait.until(
    EC.element_to_be_clickable((By.XPATH, "//a[@class='ui-state-default' and normalize-space(text())='31']"))
)
data_final.click()'''

'#Clicar no input "início"\ninput_data_inicio = wait.until(\n    EC.element_to_be_clickable((\n        By.ID, "data-inicio"\n    ))\n)\ninput_data_inicio.click()\n\n#Escolhe a data 01/01/2025 no calendário\nfor i in range(0,14): #botão vai clicar 14 vezes para voltar de hoje (março de 2026) até o mês de janeiro de 2025\n    btn_anterior = wait.until(\n        EC.element_to_be_clickable((\n            By.XPATH, "//a[@data-handler=\'prev\']"\n        ))\n    )\n    btn_anterior.click()\n\ndata_inicial = wait.until(\n    EC.element_to_be_clickable((By.XPATH, "//a[@class=\'ui-state-default\' and normalize-space(text())=\'1\']"))\n)\ndata_inicial.click()\n\n#Clicar no input "fim"\ninput_data_fim = wait.until(\n    EC.element_to_be_clickable((\n        By.ID, "data-fim"\n    ))\n)\ninput_data_fim.click()\n\n#Escolher a data 31/12/2025 no calendário\nfor i in range(0, 3): #já aqui o botão clica 3 vezes para voltar de março de 2026 (hoje) até dezembro de 2025\n    btn_anterior = wait.until(\n 

In [12]:
#para que o date picker funcione corretamente no Colab

# “O site usa um datepicker que só dispara o evento quando há interação humana.
# Em ambiente headless, esse evento às vezes não é disparado.
# Então precisamos sincronizar o valor visual com o valor real do formulário.”

driver.execute_script("""
document.getElementById('data-inicio').value = '01/01/2025';
document.getElementById('data-fim').value = '31/12/2025';
""")

driver.execute_script("""
document.getElementById('data-inicio').dispatchEvent(new Event('change'));
document.getElementById('data-fim').dispatchEvent(new Event('change'));
""") #aqui, dispara o evento no javascript

In [13]:
#Selecioar a opção "seção 3" (compras, contratos, aditivos, convênios, licitações etc)
input_secao3 = wait.until(
    EC.element_to_be_clickable((
        By.ID, "do3"
    ))
)
input_secao3.click()

In [14]:
#Clicar em Pesquisar
pesquisar = wait.until(
    EC.element_to_be_clickable((
        By.XPATH, "//button[normalize-space()='PESQUISAR']"
    ))
)
pesquisar.click()

In [15]:
#Mais filtros...
#Clicar em "Organização Principal"
btn_tipo = wait.until(
    EC.element_to_be_clickable((By.ID, "orgPrinAction"))
)
btn_tipo.click()

opcao_MEC = wait.until(
    EC.element_to_be_clickable((By.XPATH, "//a[contains(text(), 'Ministério da Educação')]"))
)
opcao_MEC.click()

In [16]:
#Clicar em "Tipo de Ato"
btn_tipo = wait.until(
    EC.element_to_be_clickable((By.ID, "artTypeAction"))
)
btn_tipo.click()

opcao_contrato = wait.until(
    EC.element_to_be_clickable((By.XPATH, "//a[contains(text(), 'Extrato de Contrato')]"))
)
opcao_contrato.click()

In [17]:
#Validar que a busca retornou resultados
resultados = wait.until(
    EC.element_to_be_clickable((
        By.CSS_SELECTOR, "p.search-total-label"
    ))
)

texto = resultados.text.strip()
print(texto)

479 resultados para "material didático"


###3. Coleta de dados


A consulta retornou uma lista paginada. Será necessário criar uma rotina para extrair os dados de cada página

In [18]:
#Armazenar uma lista com todos os links obtidos dos contratos na página
links = wait.until(
    EC.presence_of_all_elements_located((
        By.XPATH, "//div[@class='resultados-wrapper']//a"
    ))
)

In [19]:
#Importar biblioteca de regex
import re

In [20]:
#Criar objeto de regex para extrairmos os dados que queremos de cada contrato
CAMPOS = {
    "processo": r"Nº Processo:\s*(.*?)(?=Inexigibilidade|Contratante:)",
    "contratante": r"Contratante:\s*(.*?)(?=Contratado:)",
    "contratado": r"Contratado:\s*(.*?)(?=Objeto:)",
    "objeto": r"Objeto:\s*(.*?)(?=Fundamento Legal:|Vigência:)",
    "vigencia": r"Vigência:\s*(.*?)(?=Valor Total:)",
    "valor_total": r"Valor Total:\s*(.*?)(?=Data de Assinatura:)",
    "data_assinatura": r"Data de Assinatura:\s*(.*?)(?=\(|$)"
}

In [21]:
#Lista para armazenarmos cada resultado
registros = []

In [22]:
#Rotina para formatar os dados capturados conforme o padrão do regex informado
def extrair_campos(texto):
    dados = {}
    for campo, padrao in CAMPOS.items():
        registro = re.search(padrao, texto, re.IGNORECASE | re.DOTALL)
        if registro:
            dados[campo] = registro.group(1).strip().rstrip(".")
    return dados

    #O sinalizador ''DOTALL'' (também conhecido como sinalizador s ou modo “linha única”) modifica o comportamento do metacaractere ponto (.) para corresponder a qualquer caractere, incluindo terminadores de linha (como \n ou \r), aos quais normalmente não corresponderia.

In [23]:
#Clicar em cada link, salvando os dados de cada contrato na lista de registros
def clicar_links_pagina():
  #Obtem os elementos que possuem os links dos contratos
  links_contratos = wait.until(
    EC.presence_of_all_elements_located((
        By.XPATH, "//div[@class='resultados-wrapper']//a"
    ))
  )

  #Itera em cada elemento para clicar no link do contrato
  for i in range(len(links_contratos)):
    links = wait.until(
      EC.presence_of_all_elements_located((
          By.XPATH, "//div[@class='resultados-wrapper']//a"
      ))
    ) #sim, precisamos coletar de novo a cada iteração os links dos contratos
    link = links[i]

    #Abre uma nova página e aguarda 30 milisegundos para carregar textos na tela
    ActionChains(driver)\
        .move_to_element(link)\
        .pause(0.3)\
        .click()\
        .perform()

    #Extraindo dados dos documentos
    paragrafos = driver.find_elements(By.CSS_SELECTOR, "p.dou-paragraph")
    texto = " ".join(p.text for p in paragrafos)
    dados = extrair_campos(texto)
    registros.append(dados)

    #Fecha a página atual e volta à página principal de consulta
    driver.back()

    #Garante que os resultados foram carregados corretamente antes de clicar no próximo link
    wait.until(EC.presence_of_all_elements_located(
        (By.XPATH, "//div[@class='resultados-wrapper']//a")
    ))

clicar_links_pagina()

In [24]:
for registro in registros:
  print(registro)

{'processo': '23034.031501/2023-41', 'contratante': 'FUNDO NACIONAL DE DESENVOLVIMENTO DA EDUCACAO', 'contratado': '04.486.030/0001-00 - LUCCA CULTURA LTDA', 'objeto': 'Aquisição de obras literárias no âmbito do programa nacional do livro e do material didático - pnld 2023, objeto 03, nos formatos impresso, html, pdf e vídeo tutorial, destinadas aos estudantes e professores dos anos iniciais do ensino fundamental (1º ao 5º ano), das escolas da educação básica pública, das redes federal, estaduais, municipais e do distrito federal', 'vigencia': '15/12/2025 a 10/12/2026', 'valor_total': 'R$ 290.282,97', 'data_assinatura': '15/12/2025'}
{'processo': '23034.025122/2023-11', 'contratante': 'FUNDO NACIONAL DE DESENVOLVIMENTO DA EDUCACAO', 'contratado': '22.622.213/0001-98 - BAMBOLE EDITORA E LIVRARIA LTDA', 'objeto': 'Aquisição de obras literárias no âmbito do programa nacional do livro e do material didático - pnld 2023, objeto 03, nos formatos impresso, html, pdf e vídeo tutorial, destinad

In [25]:
#Limpar a lista de resultados
registros = []

In [26]:
#Descobrir qual é a última página de resultado
def ultima_pagina():
  try:
    btn_ultima_pagina = wait.until(
        EC.element_to_be_clickable((By.ID, "lastPage" ))
    )
    ultima_pagina = int(btn_ultima_pagina.text)
  except:
    ultima_pagina = 1

  print(f"Última página de resultados: {ultima_pagina}")
  return ultima_pagina

In [27]:
#Rotina para iterar da primeira até a última página de resultados, capturando os textos de cada contrato
def extrair_resultados_todas_pgs():
    total = ultima_pagina()

    for pagina in range(1, total + 1):
        print(f"Indo para a página {pagina}")

        # Coleta os contratos da página atual
        clicar_links_pagina()

        # Se ainda não é a última página, vai para a próxima
        if pagina < total:
            btn_proximo = wait.until(
                EC.element_to_be_clickable((
                    By.ID, "rightArrow"
                ))
            )

            # driver.execute_script(
            #     "arguments[0].scrollIntoView({block:'center'});",
            #     btn_proximo
            # )
            btn_proximo.click()

            # Aguarda os resultados da próxima página carregarem
            wait.until(
                EC.presence_of_element_located((
                    By.XPATH, "//div[@class='resultados-wrapper']//a"
                ))
            )

extrair_resultados_todas_pgs()

Última página de resultados: 24
Indo para a página 1
Indo para a página 2
Indo para a página 3
Indo para a página 4
Indo para a página 5
Indo para a página 6
Indo para a página 7
Indo para a página 8
Indo para a página 9
Indo para a página 10
Indo para a página 11
Indo para a página 12
Indo para a página 13
Indo para a página 14
Indo para a página 15
Indo para a página 16
Indo para a página 17
Indo para a página 18
Indo para a página 19
Indo para a página 20
Indo para a página 21
Indo para a página 22
Indo para a página 23
Indo para a página 24


In [28]:
print(f"Total de contratos: {len(registros)}")

Total de contratos: 479


In [29]:
driver.quit()

###4. Salvar resultados em .csv

In [30]:
#Importar a biblioteca Pandas
import pandas as pd

In [31]:
#Cria um DataFrame e salva o resultado como arquivo .csv
df = pd.DataFrame(registros)
df.to_csv('gastos_material_didatico_mec_2025.csv', index=False, encoding='utf-8')

In [32]:
df.head()

,processo,contratante,contratado,objeto,vigencia,valor_total,data_assinatura
0,23034.031501/2023-41,FUNDO NACIONAL DE DESENVOLVIMENTO DA EDUCACAO,04.486.030/0001-00 - LUCCA CULTURA LTDA,Aquisição de obras literárias no âmbito do pro...,15/12/2025 a 10/12/2026,"R$ 290.282,97",15/12/2025
1,23034.025122/2023-11,FUNDO NACIONAL DE DESENVOLVIMENTO DA EDUCACAO,22.622.213/0001-98 - BAMBOLE EDITORA E LIVRARI...,Aquisição de obras literárias no âmbito do pro...,15/12/2025 a 10/12/2026,"R$ 725.941,01",15/12/2025
2,23034.031153/2023-10,FUNDO NACIONAL DE DESENVOLVIMENTO DA EDUCACAO,20.513.288/0001-05 - LAG NEIRA,Aquisição de obras literárias no âmbito do pro...,15/12/2025 a 10/12/2026,"R$ 585.684,78",15/12/2025
3,23034.031498/2023-65,FUNDO NACIONAL DE DESENVOLVIMENTO DA EDUCACAO,43.401.056/0001-60 - LONGUINA EDITORA LTDA,Aquisição de obras literárias no âmbito do pro...,15/12/2025 a 10/12/2026,"R$ 242.918,22",15/12/2025
4,23034.031539/2023-13,FUNDO NACIONAL DE DESENVOLVIMENTO DA EDUCACAO,08.675.034/0001-98 - SCOPPIO EDITORA LTDA,Aquisição de obras literárias no âmbito do pro...,15/12/2025 a 10/12/2026,"R$ 201.005,82",15/12/2025


###5. Tratamento de dados

In [33]:
import math

def limpar_valor(valor):
    if valor is None:
        return None

    # Se já for int, retorna float do valor
    if isinstance(valor, (int, float)):
        if math.isnan(valor):
            return None
        return float(valor)

    # Se for string, transforma em float
    valor = str(valor)

    return float(
        valor.replace("R$", "")
             .replace(".", "")
             .replace(",", ".")
             .strip()
    )

In [34]:
df["valor_total"] = df["valor_total"].apply(limpar_valor)

In [35]:
df.head()

,processo,contratante,contratado,objeto,vigencia,valor_total,data_assinatura
0,23034.031501/2023-41,FUNDO NACIONAL DE DESENVOLVIMENTO DA EDUCACAO,04.486.030/0001-00 - LUCCA CULTURA LTDA,Aquisição de obras literárias no âmbito do pro...,15/12/2025 a 10/12/2026,290282.97,15/12/2025
1,23034.025122/2023-11,FUNDO NACIONAL DE DESENVOLVIMENTO DA EDUCACAO,22.622.213/0001-98 - BAMBOLE EDITORA E LIVRARI...,Aquisição de obras literárias no âmbito do pro...,15/12/2025 a 10/12/2026,725941.01,15/12/2025
2,23034.031153/2023-10,FUNDO NACIONAL DE DESENVOLVIMENTO DA EDUCACAO,20.513.288/0001-05 - LAG NEIRA,Aquisição de obras literárias no âmbito do pro...,15/12/2025 a 10/12/2026,585684.78,15/12/2025
3,23034.031498/2023-65,FUNDO NACIONAL DE DESENVOLVIMENTO DA EDUCACAO,43.401.056/0001-60 - LONGUINA EDITORA LTDA,Aquisição de obras literárias no âmbito do pro...,15/12/2025 a 10/12/2026,242918.22,15/12/2025
4,23034.031539/2023-13,FUNDO NACIONAL DE DESENVOLVIMENTO DA EDUCACAO,08.675.034/0001-98 - SCOPPIO EDITORA LTDA,Aquisição de obras literárias no âmbito do pro...,15/12/2025 a 10/12/2026,201005.82,15/12/2025


In [36]:
df[df.isna().any(axis=1)] #mostra as linhas com valor NaN

,processo,contratante,contratado,objeto,vigencia,valor_total,data_assinatura
36,NaN,NaN,NaN,NaN,NaN,NaN,NaN
80,NaN,NaN,NaN,NaN,NaN,NaN,NaN
93,NaN,NaN,NaN,NaN,NaN,NaN,NaN
103,NaN,NaN,NaN,NaN,NaN,NaN,NaN
138,NaN,NaN,NaN,NaN,NaN,NaN,NaN
214,NaN,NaN,NaN,NaN,NaN,NaN,NaN
219,NaN,NaN,NaN,NaN,NaN,NaN,NaN
221,NaN,NaN,NaN,NaN,NaN,NaN,NaN
235,NaN,NaN,NaN,NaN,NaN,NaN,NaN
244,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
print(f"Total de contratos: {len(df)}") #quantas linhas tinham no dataset antes de remover os NaN


Total de contratos: 479


In [39]:
df = df.dropna() #remove linhas com NaN
print(f"Total de contratos_atualizado: {len(df)}") #quantas linhas passaram a ter no dataset depois de remover os NaN

Total de contratos_atualizado: 452


Salvando o dataset final em csv

In [82]:
#para que o dataset funcione perfeitamente em Looker, PowerBI, Excel, Sheets, garantir que as datas estejam como datetime
df["data_assinatura"] = pd.to_datetime(df["data_assinatura"])

In [90]:
# salvar dataframe como csv
df.to_csv("contratos_mec_2025.csv", index=False, encoding="utf-8-sig")

# gerar botão de download no Colab
from google.colab import files
files.download("contratos_mec_2025.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##6. Análise de dados

1. Como estão distribuídos os valores dos 452 contratos?

In [40]:
describe_valores = df["valor_total"].describe()

describe_formatado = describe_valores.apply(
    lambda x: f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)

print(describe_formatado)

count            R$ 452,00
mean       R$ 2.184.797,46
std       R$ 15.538.442,20
min            R$ 7.000,00
25%          R$ 308.634,75
50%          R$ 496.272,21
75%          R$ 612.283,00
max      R$ 223.685.779,92
Name: valor_total, dtype: object


2. Qual foi o gasto total em materiais didáticos?

In [41]:
valor_contratos_2025 = df['valor_total'].sum()

print(
    f"O MEC adquiriu R$ {valor_contratos_2025:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."),
    "em materiais didáticos no ano de 2025")

O MEC adquiriu R$ 987.528.452,58 em materiais didáticos no ano de 2025


3. Qual é o valor médio dos contratos?

In [42]:
media_contratos = df["valor_total"].mean()

media_formatada = f"{media_contratos:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

print(f"Valor médio dos contratos: R$ {media_formatada}")

Valor médio dos contratos: R$ 2.184.797,46


4.1 Quais foram os contratos mais caros?

In [45]:
# calcular valor total por empresa
valor_por_empresa = df.groupby("contratado")["valor_total"].sum()

# selecionar top 10
top_empresas = valor_por_empresa.sort_values(ascending=False).head(10)

top_empresas_formatado = top_empresas.apply(
    lambda x: f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)

print(top_empresas_formatado)

contratado
61.186.490/0001-57 - EDITORA FTD S A                                            R$ 224.304.108,38
62.136.304/0001-38 - EDITORA MODERNA LTDA                                       R$ 166.349.489,09
61.259.958/0001-96 - EDITORA ATICA S.A                                          R$ 141.299.720,82
60.657.574/0001-69 - EDITORA DO BRASIL SA                                       R$ 100.961.692,23
44.127.355/0001-11 - EDITORA SCIPIONE S.A                                        R$ 41.837.200,27
50.268.838/0001-39 - SARAIVA EDUCACAO S.A                                        R$ 36.419.411,85
15.194.990/0001-13 - UNIVERSO EDUCACAO LTDA                                      R$ 12.477.798,44
61.016.028/0001-01 - IBEP - INSTITUTO BRASILEIRO DE EDICOES PEDAGOGICAS LTDA     R$ 10.783.523,12
11.690.588/0001-79 - ATELIE DA ESCRITA EDITORA LTDA                               R$ 7.709.963,93
21.044.458/0001-12 - EDITORA DIMENSAO LTDA                                        R$ 6.905.416,43
Name: val

4.2. Qual foi o contrato mais barato?

In [46]:
contrato_mais_barato = df.loc[df["valor_total"].idxmin()]

valor_formatado = f"{contrato_mais_barato['valor_total']:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

print("Contrato mais barato:")
print("Empresa:", contrato_mais_barato["contratado"])
print("Valor: R$", valor_formatado)
print("Processo:", contrato_mais_barato["processo"])

Contrato mais barato:
Empresa: 09.529.916/0001-08 - SIMULARE SISTEMAS DE INFORMACOES LTDA
Valor: R$ 7.000,00
Processo: 23430.001705/2025-46


5. Quantas empresas diferentes participaram?

In [48]:
numero_empresas = df["contratado"].nunique()

print("Número de empresas contratadas:", numero_empresas)

Número de empresas contratadas: 435


5.1. Quais foram os licitantes mais selecionados?

In [49]:
licitantes_mais_selecionados = df["contratado"].value_counts()

print(licitantes_mais_selecionados.head(5))

contratado
11.690.588/0001-79 - ATELIE DA ESCRITA EDITORA LTDA    3
44.127.355/0001-11 - EDITORA SCIPIONE S.A              2
50.268.838/0001-39 - SARAIVA EDUCACAO S.A              2
61.186.490/0001-57 - EDITORA FTD S A                   2
61.259.958/0001-96 - EDITORA ATICA S.A                 2
Name: count, dtype: int64


5.2. Concentração de fornecedores (HHI)

Para avaliar se as compras estão concentradas em poucos fornecedores, foi calculado o Índice Herfindahl-Hirschman (HHI).<br><br>

O HHI é um indicador utilizado em economia para medir concentração de mercado. Ele é calculado a partir da soma dos quadrados das participações de mercado de cada fornecedor.

Quanto maior o índice, maior a concentração em poucos fornecedores.<br><br>

Interpretação padrão:

HHI	< 1500	= baixa concentração <br>
HHI > 1500 e < 2500	= concentração moderada <br>
HHI > 2500	= alta concentração<br><br>


In [50]:
valor_por_empresa = df.groupby("contratado")["valor_total"].sum()

In [51]:
participacao = valor_por_empresa / valor_por_empresa.sum()

In [54]:
hhi = (participacao ** 2).sum() * 10000

print("HHI:", round(hhi,2))

if hhi < 1500:
    print("Baixa concentração de fornecedores")
elif hhi < 2500:
    print("Concentração moderada")
else:
    print("Alta concentração de fornecedores")

HHI: 1147.35
Baixa concentração de fornecedores


6. Qual mês teve mais contratos?

In [55]:
df["data_assinatura"] = pd.to_datetime(df["data_assinatura"], format="%d/%m/%Y")
df["mes"] = df["data_assinatura"].dt.month

contratos_por_mes = df["mes"].value_counts().sort_index()

print(contratos_por_mes)

mes
8       8
9       7
11      1
12    436
Name: count, dtype: int64


6.1. Qual mês teve maior volume financeiro?

In [56]:
valor_por_mes = df.groupby("mes")["valor_total"].sum().sort_values(ascending=False)

valor_por_mes_df = valor_por_mes.reset_index()
valor_por_mes_df.columns = ["mes", "valor_total"]

valor_por_mes_df["valor_total"] = valor_por_mes_df["valor_total"].apply(
    lambda x: f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)

print(valor_por_mes_df)

   mes        valor_total
0    8  R$ 731.126.010,49
1   12  R$ 226.810.932,50
2    9   R$ 29.584.509,59
3   11        R$ 7.000,00
